# 1D CNN for TAG Authentication — Physical Layer (v3)

**Input**: Raw equalized signal `y/h` — vector of length 1024  
**Architecture**: 1D CNN (Chin & Chin, Eng. Proc. 2025) adapted for L=1024  
**Theory**: Braca et al. (2022) D3F framework — ML learns T^(n) converging to LLR  
**Threshold**: ROC → maximize PD subject to FPR ≤ α (Neyman-Pearson constrained)

```
Input (1024, 1)
  → Conv1D(32, k=15) + BatchNorm + ReLU + MaxPool(4)   [→ 256, 32]
  → Conv1D(64,  k=9) + BatchNorm + ReLU + MaxPool(4)   [→  64, 64]
  → Conv1D(128, k=5) + BatchNorm + ReLU + GlobalAvgPool[→ 128]
  → Dense(64, ReLU) + Dropout(0.3)
  → Dense(1, Sigmoid)  →  P(H1)
```

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import os
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, confusion_matrix, roc_curve
)
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
import tensorflow.keras.backend as K

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# ============================================================================
# SETUP PATHS FOR NEW DIRECTORY STRUCTURE
# ============================================================================
from pathlib import Path

notebook_dir = Path.cwd()  # Current: notebooks/
project_root = notebook_dir.parent  # Go up to: Redes Neurais/
results_dir = project_root / "results"
data_dir = results_dir / "data"
models_dir = results_dir / "models"
visualizations_dir = results_dir / "visualizations"

# Ensure directories exist
models_dir.mkdir(parents=True, exist_ok=True)
visualizations_dir.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {data_dir}")
print(f"Models directory: {models_dir}")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)


TensorFlow version: 2.21.0
GPU available: []
Data directory: c:\Users\thami\OneDrive\Documents\TAG-Authentication\Redes Neurais\results\data
Models directory: c:\Users\thami\OneDrive\Documents\TAG-Authentication\Redes Neurais\results\models


In [2]:
# ==============================================================================
# 2. LOAD DATASET (CNN y/h vectors)
# ==============================================================================

dataset_path = data_dir / "dataset_cnn_yeq_0_30dB.h5"

if not dataset_path.exists():
    raise FileNotFoundError(
        f"{dataset_path} not found.\nRun NN_01_DataGeneration.ipynb first."
    )

with h5py.File(str(dataset_path), 'r') as f:
    # y_eq: (N, 1024) float32 — CNN input
    X_train_raw = f['train/y_eq'][:]
    y_train     = f['train/y'][:]
    snr_train   = f['train/snr'][:]

    X_val_raw   = f['val/y_eq'][:]
    y_val       = f['val/y'][:]
    snr_val     = f['val/snr'][:]

    X_test_raw  = f['test/y_eq'][:]
    y_test      = f['test/y'][:]
    snr_test    = f['test/snr'][:]

    L_FIXED = int(f.attrs['L_FIXED'])

# Reshape to (N, L, 1) for Conv1D
X_train = X_train_raw.reshape(-1, L_FIXED, 1)
X_val   = X_val_raw.reshape(-1, L_FIXED, 1)
X_test  = X_test_raw.reshape(-1, L_FIXED, 1)

print("Dataset loaded:")
print(f"  Train : {X_train.shape}  labels={np.bincount(y_train)}")
print(f"  Val   : {X_val.shape}  labels={np.bincount(y_val)}")
print(f"  Test  : {X_test.shape}  labels={np.bincount(y_test)}")
print(f"  SNR range: [{snr_test.min():.1f}, {snr_test.max():.1f}] dB")

Dataset loaded:
  Train : (280000, 1024, 1)  labels=[140000 140000]
  Val   : (35000, 1024, 1)  labels=[17500 17500]
  Test  : (35000, 1024, 1)  labels=[17500 17500]
  SNR range: [0.0, 30.0] dB


In [3]:
# ==============================================================================
# 3. 1D CNN ARCHITECTURE  (Chin & Chin 2025 + Braca 2022 D3F)
# ==============================================================================

def build_cnn_1d(L=1024):
    """1D CNN binary classifier for y/h signal vectors.
    
    Architecture from Chin & Chin (Eng. Proc. 2025), scaled to L=1024:
      Conv1D(32, k=15) → Conv1D(64, k=9) → Conv1D(128, k=5) → Dense(64) → Dense(1)
    BatchNorm after each Conv layer for training stability.
    GlobalAveragePooling collapses temporal dimension → length-invariant features.
    """
    inp = layers.Input(shape=(L, 1), name='y_eq_input')

    # Block 1
    x = layers.Conv1D(32, kernel_size=15, padding='same', use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(pool_size=4)(x)          # 1024 → 256

    # Block 2
    x = layers.Conv1D(64, kernel_size=9, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(pool_size=4)(x)          # 256 → 64

    # Block 3
    x = layers.Conv1D(128, kernel_size=5, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling1D()(x)           # (64, 128) → (128,)

    # Classifier head
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid', name='P_H1')(x)

    return keras.Model(inputs=inp, outputs=out, name='CNN1D_TAG_Auth')


model = build_cnn_1d(L=L_FIXED)
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=BinaryCrossentropy(),
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)
model.summary()

Model: "CNN1D_TAG_Auth"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ y_eq_input (InputLayer)         │ (None, 1024, 1)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 1024, 32)       │           480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1024, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 1024, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 256, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 256, 64)        │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 256, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 64, 128)        │        40,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ P_H1 (Dense)                    │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 69,089 (269.88 KB)

 Trainable params: 68,641 (268.13 KB)

 Non-trainable params: 448 (1.75 KB)

In [4]:
# ==============================================================================
# 4. TRAINING CALLBACKS
# ==============================================================================

model_save_path = str(models_dir / 'cnn1d_tag_auth_best.keras')

cbs = [
    callbacks.EarlyStopping(
        monitor='val_auc', mode='max', patience=12,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ModelCheckpoint(
        model_save_path, monitor='val_auc', mode='max',
        save_best_only=True, verbose=0
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5,
        min_lr=1e-6, verbose=1
    ),
]

print(f"Model will be saved to: {model_save_path}")

Model will be saved to: c:\Users\thami\OneDrive\Documents\TAG-Authentication\Redes Neurais\results\models\cnn1d_tag_auth_best.keras


In [ ]:
# ==============================================================================
# 5. TRAIN 1D CNN
# ==============================================================================

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=256,
    callbacks=cbs,
    verbose=1
)

print(f"\nBest val AUC : {max(history.history['val_auc']):.5f}")
print(f"Best val loss: {min(history.history['val_loss']):.5f}")

Epoch 1/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 179s 155ms/step - accuracy: 0.7089 - auc: 0.8070 - loss: 0.5081 - val_accuracy: 0.7548 - val_auc: 0.8634 - val_loss: 0.4353 - learning_rate: 0.0010
Epoch 2/100
 526/1094 ━━━━━━━━━━━━━━━━━━━━ 3:54 413ms/step - accuracy: 0.7641 - auc: 0.8695 - loss: 0.4269

KeyboardInterrupt: 

: 

In [ ]:
# ==============================================================================
# 6. EVALUATION WITH α-CONSTRAINED THRESHOLD
# ==============================================================================

# Scores on all sets
p_val  = model.predict(X_val,  batch_size=512, verbose=0).flatten()
p_test = model.predict(X_test, batch_size=512, verbose=0).flatten()

# --- ROC on validation set ---
fpr_v, tpr_v, thr_v = roc_curve(y_val, p_val)
auc_val = roc_auc_score(y_val, p_val)
print(f"Val AUC: {auc_val:.5f}")

# --- α-constrained threshold (from val ROC) ---
# Among all thresholds where empirical FPR ≤ α_target, pick the one with max TPR.
# Note: true α=10^-7 requires ~10^8 H0 samples; here α_target is the best achievable
# empirically (N_H0_val ≈ half val size). NN_07 will do the full Monte Carlo comparison.
alpha_target = 1e-3   # best empirical constraint with ~17k H0 val samples

valid_mask = fpr_v <= alpha_target
if valid_mask.any():
    best_idx   = np.argmax(tpr_v[valid_mask])
    best_thr   = thr_v[valid_mask][best_idx]
    best_tpr   = tpr_v[valid_mask][best_idx]
    best_fpr   = fpr_v[valid_mask][best_idx]
else:
    # fallback: pick lowest achievable FPR threshold
    best_idx = np.argmin(fpr_v[1:]) + 1
    best_thr = thr_v[best_idx]
    best_tpr = tpr_v[best_idx]
    best_fpr = fpr_v[best_idx]

print(f"\nα-constrained threshold (α_target={alpha_target:.0e}):")
print(f"  Threshold  : {best_thr:.5f}")
print(f"  FPR (α)    : {best_fpr:.2e}")
print(f"  PD (=TPR)  : {best_tpr:.5f}  (= 1 − FNR)")

# --- Test set metrics at constrained threshold ---
y_pred_constrained = (p_test >= best_thr).astype(int)
cm = confusion_matrix(y_test, y_pred_constrained)
tn, fp, fn, tp = cm.ravel()

fpr_test = fp / (fp + tn)
pd_test  = tp / (tp + fn)
fnr_test = fn / (fn + tp)

print(f"\nTest set @ constrained threshold ({best_thr:.5f}):")
print(f"  FPR (α)    : {fpr_test:.2e}")
print(f"  PD         : {pd_test:.5f}")
print(f"  FNR (β)    : {fnr_test:.5f}")
print(f"  AUC        : {roc_auc_score(y_test, p_test):.5f}")

# --- PD vs SNR on test set (using constrained threshold) ---
snr_bins_plot = np.arange(0, 31, 5)
pd_vs_snr     = []

for snr_lo in snr_bins_plot[:-1]:
    snr_hi = snr_lo + 5
    mask_h1 = (snr_test >= snr_lo) & (snr_test < snr_hi) & (y_test == 1)
    if mask_h1.sum() < 10:
        pd_vs_snr.append(np.nan)
        continue
    p_h1   = p_test[mask_h1]
    pd_bin = float((p_h1 >= best_thr).mean())
    pd_vs_snr.append(pd_bin)

snr_centres = snr_bins_plot[:-1] + 2.5
print(f"\nPD vs SNR (constrained threshold, α≤{alpha_target:.0e}):")
for s, pd in zip(snr_centres, pd_vs_snr):
    bar = '#' * int((pd or 0) * 40)
    print(f"  SNR {s:4.0f} dB  PD={pd:.4f}  {bar}")

In [ ]:
# ==============================================================================
# 7. VISUALIZATIONS
# ==============================================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1: Loss curve
ax = axes[0, 0]
ax.plot(history.history['loss'],     label='Train', linewidth=2)
ax.plot(history.history['val_loss'], label='Val',   linewidth=2)
ax.set(xlabel='Epoch', ylabel='Loss', title='Training Loss')
ax.legend(); ax.grid(alpha=0.3)

# 2: AUC curve
ax = axes[0, 1]
ax.plot(history.history['auc'],     label='Train', linewidth=2)
ax.plot(history.history['val_auc'], label='Val',   linewidth=2)
ax.set(xlabel='Epoch', ylabel='AUC', title='Training AUC')
ax.legend(); ax.grid(alpha=0.3)

# 3: ROC with constrained operating point
ax = axes[0, 2]
fpr_t, tpr_t, _ = roc_curve(y_test, p_test)
ax.plot(fpr_t, tpr_t, linewidth=2, label=f'CNN (AUC={roc_auc_score(y_test, p_test):.4f})')
ax.scatter([fpr_test], [pd_test], s=100, zorder=5, color='red',
           label=f'α≤{alpha_target:.0e} point\n(τ={best_thr:.4f})')
ax.axvline(alpha_target, color='gray', linestyle='--', alpha=0.6, label=f'α={alpha_target:.0e}')
ax.plot([0,1],[0,1],'k--', alpha=0.3)
ax.set(xlabel='FPR (α)', ylabel='PD (TPR)', title='ROC — Test Set')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# 4: P(H1) distribution
ax = axes[1, 0]
ax.hist(p_test[y_test == 0], bins=60, alpha=0.6, label='H0 (attacker)',   color='tomato',   density=True)
ax.hist(p_test[y_test == 1], bins=60, alpha=0.6, label='H1 (authentic)',  color='steelblue', density=True)
ax.axvline(best_thr, color='k', linestyle='--', linewidth=1.5, label=f'τ*={best_thr:.4f}')
ax.set(xlabel='P(H1)', ylabel='Density', title='Score Distribution (Test)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# 5: PD vs SNR
ax = axes[1, 1]
pd_arr = np.array(pd_vs_snr, dtype=float)
ax.plot(snr_centres, pd_arr, 'o-', linewidth=2, markersize=6, color='steelblue', label='1D CNN')
ax.axhline(1.0, color='gray', linestyle=':', alpha=0.5)
ax.set(xlabel='SNR (dB)', ylabel='PD', title=f'PD vs SNR (α≤{alpha_target:.0e})',
       ylim=(-0.05, 1.05), xlim=(-1, 31))
ax.legend(); ax.grid(alpha=0.3)

# 6: Confusion matrix at constrained threshold
ax = axes[1, 2]
cm_disp = confusion_matrix(y_test, y_pred_constrained)
im = ax.imshow(cm_disp, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm_disp[i, j], ha='center', va='center',
                color='white' if cm_disp[i, j] > cm_disp.max()/2 else 'black', fontsize=13)
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['H0 pred','H1 pred']); ax.set_yticklabels(['H0 true','H1 true'])
ax.set_title(f'Confusion Matrix (τ*={best_thr:.4f})')

plt.tight_layout()
vis_path = visualizations_dir / "NN02_CNN1D_results.png"
plt.savefig(str(vis_path), dpi=120)
plt.show()
print(f"Saved → {vis_path}")

In [ ]:
# ==============================================================================
# 8. SAVE MODEL + THRESHOLD + METRICS
# ==============================================================================

import json

# Save threshold for use in NN_07
threshold_data = {
    'cnn_threshold': float(best_thr),
    'alpha_target':  float(alpha_target),
    'fpr_at_thr':    float(fpr_test),
    'pd_at_thr':     float(pd_test),
    'auc':           float(roc_auc_score(y_test, p_test)),
    'pd_vs_snr':     {f"snr_{int(s)}": (float(p) if not np.isnan(p) else None)
                      for s, p in zip(snr_centres, pd_vs_snr)},
}

thr_path = models_dir / 'cnn1d_threshold.json'
with open(str(thr_path), 'w') as f:
    json.dump(threshold_data, f, indent=2)

print(f"Saved model   → {model_save_path}")
print(f"Saved threshold → {thr_path}")
print(f"\nKey results:")
print(f"  AUC        : {threshold_data['auc']:.5f}")
print(f"  Threshold  : {best_thr:.5f}  (α≤{alpha_target:.0e})")
print(f"  PD (test)  : {pd_test:.5f}")
print(f"  FPR (test) : {fpr_test:.2e}")

## Summary

The 1D CNN (Chin & Chin 2025) receives the raw equalized signal `y/h` and learns to distinguish:
- **H1**: `y/h = ρ_s·msg + ρ_t·tag + w/h` — authentic TAG present
- **H0**: `y/h = ρ_s·msg + w/h` — unauthenticated (no TAG)

Under H0 the residual `(y/h − ρ_s·msg)` is pure noise, while under H1 it carries the TAG component `ρ_t·tag`. The 1D CNN learns this structure directly from the raw signal; the classical correlator (Xie 2021) exploits it explicitly via inner product with `tag_ref`.

The threshold is set via the validation ROC curve to satisfy a false positive constraint (α ≤ target), following the Neyman-Pearson constrained paradigm from Braca et al. (2022).

**Next**: NN_07 — generate the revised Figure 2 (PD vs SNR) comparing this 1D CNN against the classical Xie 2021 correlator, both operating under the same α = 10⁻⁷ constraint via large-N Monte Carlo simulation.